# Stellar Class — Kaggle Playground Series S6E6

The [**Sloan Digital Sky Survey (SDSS)**](https://en.wikipedia.org/wiki/Sloan_Digital_Sky_Survey) photographed the sky in five colour bands and measured the spectra of millions of astronomical objects. This notebook works on the Kaggle *Playground Series* version of that catalogue.

<center>
<img src="https://upload.wikimedia.org/wikipedia/commons/8/82/SDSS_Camera.jpg" width="300">
<br>
<em>(The SDSS Imaging Camera, the heart of the 2.5m Telescope. Source: Wikimedia Commons)</em>
</center>

## The task

Given a handful of photometric measurements, predict which of three kinds of object we are looking at:

| Column | Role | Description |
| :--- | :--- | :--- |
| `u, g, r, i, z` | feature | brightness (magnitude) in five colour filters |
| `redshift` | feature | how fast the object recedes from us; stars are ≈ 0 |
| `alpha, delta` | feature | sky coordinates |
| `spectral_type`, `galaxy_population` | feature | categorical metadata |
| `class` | **target** | **STAR**, **GALAXY** or **QSO** (quasar) |

The training set has 577,347 rows and the test set 247,435.

## The metric: balanced accuracy (not plain accuracy!)

This is the single most important fact in the whole notebook. Kaggle scores this competition with

$$\text{Balanced Accuracy} = \frac{1}{C}\sum_{c=1}^{C}\frac{TP_c}{TP_c+FN_c} = \frac{1}{C}\sum_{c=1}^{C}\text{Recall}_c$$

In words: **compute the recall of each class separately, then average them.** Every class counts equally, however rare it is.

Why does that matter? Look at the label distribution:

| Class | Count | Share |
| :--- | ---: | ---: |
| GALAXY | 377,480 | 65.4 % |
| QSO | 117,143 | 20.3 % |
| STAR | 82,724 | **14.3 %** |

A lazy model that answers "GALAXY" for *everything* scores **65 % accuracy** but only **33 % balanced accuracy**. Plain accuracy rewards ignoring the rare STAR class; balanced accuracy refuses to. Everything below is built around it: we weight the loss, we *select* checkpoints by balanced accuracy, and we report `accuracy`, `balanced_acc` and `macro_f1` side by side.

## How this notebook is organised

We start by implementing the classic algorithms **from scratch** (no library magic) to build intuition, and then switch to a full competition pipeline with deep models.

| Section | Topic | One-line idea |
| :--- | :--- | :--- |
| **1** | Data preprocessing & visualisation | encode labels, min–max normalise, look at the redshift distribution |
| **2** | Decision stump | one rule: a redshift threshold for STAR vs non-STAR |
| **3** | K-nearest neighbours | multi-class neighbour voting |
| **4** | K-means | unsupervised structure and the elbow method |
| **5** | Per-class metrics | precision / recall / F1 for each class |
| — | Classic baseline | run all of the above on the real competition data |
| **6** | Competition setup | metric, leakage-free split, feature engineering, shared trainer |
| **7** | Gradient-boosted trees | the strongest classical model for tabular data |
| **8** | MLP | stacked neurons trained with back-propagation |
| **9** | RNN | treat the features as a *sequence* and carry a hidden state |
| **10** | LSTM | gated memory that fixes the RNN's vanishing gradient |
| **11** | Transformer | self-attention over feature "tokens" |
| **12** | GAN | synthesise minority-class samples and fight the imbalance |
| **13** | Ensemble | blend every model into the final `submission.csv` |
| **14** | Wrapping up | results, lessons, extensions |

> **Running this in Colab.** Put `train.csv` and `test.csv` next to the notebook (or mount your Drive) and select **Runtime → Change runtime type → GPU (T4)**. The notebook finds the files automatically and tells you which device it is using.

> **Two feature sets.** Sections 1–5 use a small feature set (8 columns) so the from-scratch algorithms stay readable. From Section 6 onwards we use a richer, engineered feature set (14 columns) for the models that actually compete.


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Dict, Tuple, List

plt.rcParams["figure.dpi"] = 110

# --- Locate the competition CSVs (works on Colab and locally) ----------
CANDIDATE_DIRS = [".", "playground-series-s6e6", "/content",
                  "/content/playground-series-s6e6", "/content/drive/MyDrive"]

def find_csv(name):
    for d in CANDIDATE_DIRS:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    hits = glob.glob(os.path.join("**", name), recursive=True)
    if hits:
        return hits[0]
    raise FileNotFoundError(f"Could not find {name}. Put {name} next to the notebook.")

# --- A small sample that every visualisation below reuses --------------
SAMPLE_N = 50_000
sample_df = pd.read_csv(find_csv("train.csv"), nrows=SAMPLE_N)
print("sample for the plots:", sample_df.shape)
display(sample_df.head())


## Section 1 — Data Preprocessing & Visualisation

The `class` column holds strings and every model wants integers, so we encode `STAR → 0`, `GALAXY → 1`, `QSO → 2`. The features live on very different scales, so we also **min–max normalise** them onto $[0, 1]$:

$$x_{norm} = \frac{x - \min(x)}{\max(x) - \min(x)}$$

Two details matter:

* **Data leakage.** The min/max must be computed on the training data only and then applied to the test data. Computing them over everything at once would quietly leak test information into the model. That is why `preprocess_data` takes a `stats` dictionary: the first call (on the training frame) computes and returns it, and later calls (on the test frame) reuse it.
* **Division by zero.** If a feature is constant on the training data then `max - min = 0`, which would divide by zero; such a column falls back to 0.

We also visualise the `redshift` distribution of each class. This one plot explains why the decision stump of Section 2 works at all: stars sit at redshift ≈ 0 while galaxies and quasars spread over much larger values.


In [ ]:
def preprocess_data(df: pd.DataFrame, stats: Optional[Dict[str, Dict[str, float]]] = None) -> Tuple[np.ndarray, np.ndarray, Dict[str, Dict[str, float]]]:
    """
    Preprocesses the dataset by encoding labels and normalizing features.

    Args:
        df: Input DataFrame.
        stats: Dictionary containing min/max for normalization (optional).

    Returns:
        X: Processed features as a numpy array of shape (N, D).
        y: Labels as a numpy array of shape (N,).
        stats: Dictionary containing the train-set min/max used for normalization.
    """
    df = df.copy()
    # --- Label Encoding ---
    class_map = {'STAR': 0, 'GALAXY': 1, 'QSO': 2}
    spectral_map = {'M': 0, 'A/F': 0.25, 'O/B': 0.5,"G/K":0.75}
    population_map={'Red_Sequence':0,"Blue_Cloud":2}

    # 1. Apply class mapping to 'class' column
    if df.shape[1]==12:
      df["class"]=df["class"].map(class_map)
    df["spectral_type"]=df["spectral_type"].map(spectral_map)
    df["galaxy_population"]=df["galaxy_population"].map(population_map)
    # --- Min-Max Normalization ---
    feature_cols = ['alpha','delta','u', 'g', 'r', 'i', 'z', 'redshift']
    # 2. Compute or use existing stats for normalization
    #    If stats is None, compute min/max for each column in feature_cols and store in stats.
    #    Structure of stats: {'col_name': {'min': val, 'max': val}, ...}
    if stats is None:
      stats={}
      for col in feature_cols:
        stats[col]={'min':df[col].min(),'max':df[col].max()}
    # 3. Apply normalization to feature_cols
    for col in feature_cols:
      if stats[col]['max']-stats[col]['min']!=0:
        df[col]=(df[col]-stats[col]['min'])/(stats[col]['max']-stats[col]['min'])
      else:
        df[col]=0
    df["redshift"]*=3
    # 4. Extract X (features) and y (labels)
    # X should be a numpy array of shape (N, D)
    # y should be a numpy array of shape (N,)

    X = np.array(df[feature_cols].values)
    if df.shape[1]==12:
      y=np.array(df['class'])
    else:
      y=np.array([])

    return X, y, stats

def plot_redshift_distribution(df: pd.DataFrame) -> None:
    """
    Plots the distribution (histogram) of redshift for each class.

    Args:
        df: The original DataFrame containing 'redshift' and 'class'.
    """
    # 1. Iterate through the 3 classes (STAR, GALAXY, QSO)
    # Define common bins for comparison
    d_star=df[df['class']=="STAR"]
    d_galaxy=df[df['class']=="GALAXY"]
    d_qso=df[df['class']=="QSO"]
    bins=np.linspace(df['redshift'].min(),df['redshift'].max())
    # 2. Plot a histogram of redshift for that class (use alpha=0.5 for transparency)
    # Use log=True to visualize imbalanced classes (QSO vs Star)
    plt.hist(d_star['redshift'],bins=bins,alpha=0.5,label='STAR',log=True)
    plt.hist(d_galaxy['redshift'],bins=bins,alpha=0.5,label='GALAXY',log=True)
    plt.hist(d_qso['redshift'],bins=bins,alpha=0.5,label='QSO',log=True)
    # 3. Add legend and labels
    plt.title('Redshift Distribution by Class (Log Scale)')
    plt.xlabel('Redshift')
    plt.ylabel('Density (Log Scale)')
    plt.legend()


### Visual check: class balance and normalisation

Two things worth seeing: how imbalanced the labels are (this is exactly why balanced accuracy matters), and what min-max normalisation does to a feature.

In [ ]:
# ---------------------------------------------------------------------
# Visual check: label balance and the effect of min-max normalisation
# ---------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = sample_df["class"].value_counts().reindex(["STAR", "GALAXY", "QSO"])
axes[0].bar(counts.index, counts.values, color=["#55A868", "#4C72B0", "#DD8452"])
for x, v in enumerate(counts.values):
    axes[0].text(x, v, f"{v / len(sample_df):.1%}", ha="center", va="bottom")
axes[0].set_title("Class distribution (imbalanced!)")
axes[0].set_ylabel("count")

raw = sample_df["redshift"].to_numpy()
normalised = (raw - raw.min()) / (raw.max() - raw.min())
axes[1].hist(raw, bins=60, alpha=0.6, label="raw")
axes[1].hist(normalised, bins=60, alpha=0.6, label="min-max normalised")
axes[1].set_title("redshift before / after normalisation")
axes[1].legend()

plt.tight_layout()
plt.show()


## Section 2 — The Decision Stump (STAR vs non-STAR)

<center>
<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/e/e4/Redshift_blueshift.svg/800px-Redshift_blueshift.svg.png" width="600">
<br>
<em>(Redshift and Blueshift. Source: Wikimedia Commons)</em>
</center>

A decision stump is the simplest possible model: a single rule, on a single feature, with a single threshold.

### The physics

Stars belong to our own galaxy and barely move relative to us, so their redshift is ≈ 0. Distant galaxies and quasars recede quickly (Hubble's law) and show a much larger redshift. A single threshold on `redshift` therefore already separates STAR from everything else:

$$\hat{y} = \begin{cases} \text{STAR }(0) & \text{if } \text{redshift} < T \\ \text{non-STAR }(1) & \text{if } \text{redshift} \ge T \end{cases}$$

*For this step we merge GALAXY $(1)$ and QSO $(2)$ into one "non-STAR" $(1)$ class.*

### Finding the optimal threshold

Sort the samples by redshift and sweep the split point from left to right, keeping the threshold with the highest accuracy. The implementation below does this in a single vectorised pass: a `np.cumsum` prefix sum scores **every** candidate threshold at once, instead of recomputing the accuracy for each one. Only the positions where the redshift value actually changes are kept as candidate thresholds.


In [ ]:
import numpy as np
class DecisionStump:
    def __init__(self):
        self.threshold = None
        self.feature_idx = -1  # 'redshift' is always the last column of X

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        """
        Finds the optimal threshold T for the redshift feature to classify Star (0) vs Non-Star (>0).
        """
        # 1. 二值化标签：0 → 0 (Star), 1/2 → 1 (Non-Star)
        y_binary = (y != 0).astype(int)
        # 2. 提取 redshift 列
        x = X[:, self.feature_idx]

        # 3. 按特征值排序
        sort_idx = np.argsort(x)
        x_sorted = x[sort_idx]
        y_sorted = y_binary[sort_idx]

        N = len(y_sorted)
        total_ones = np.sum(y_sorted)

        # 4. 前缀和（方便快速计算左边/右边 1 的个数）
        cumsum = np.cumsum(y_sorted)   # cumsum[i] = sum(y_sorted[0..i])

        # 5. 计算每个分割位置（左子包含 0..i-1，右子包含 i..N-1）的正确分类数
        # 正确数 = 左边 0 的个数 + 右边 1 的个数
        # 左边 0 的个数 = i - (左边 1 的个数) = i - cumsum[i-1]（i>0 时）
        # 右边 1 的个数 = total_ones - cumsum[i-1]
        left_zeros = np.zeros(N, dtype=int)
        right_ones = np.zeros(N, dtype=int)

        # i = 0 的情况（阈值取最小值）
        left_zeros[0] = 0
        right_ones[0] = total_ones

        # i >= 1 的情况（利用数组切片一次性计算）
        left_zeros[1:] = np.arange(1, N) - cumsum[:-1]   # i - cumsum[i-1]
        right_ones[1:] = total_ones - cumsum[:-1]

        correct = left_zeros + right_ones   # 每个位置作为分割点时正确分类的样本数

        # 6. 只保留特征值发生变化的位置作为候选阈值（与原代码用 np.unique 的语义一致）
        # 特征值重复时，第一个出现的位置才是一个有效的不同阈值
        is_unique_start = np.concatenate(([True], x_sorted[1:] != x_sorted[:-1]))
        unique_indices = np.where(is_unique_start)[0]

        # 取出候选阈值及其对应的正确数
        candidate_thresholds = x_sorted[unique_indices]
        candidate_correct = correct[unique_indices]

        # 7. 找到正确数最大的阈值（若并列，取第一个，与原循环行为一致）
        best_idx = np.argmax(candidate_correct)
        self.threshold = candidate_thresholds[best_idx]
        # 注：原代码没有保存 max_acc，这里也不需要额外存储
    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        Predicts binary class (0=Star, 1=Non-Star) using the learned threshold.

        Args:
            X: Feature array of shape (M, D).

        Returns:
            preds: Array of binary predictions (0 or 1) of shape (M,).
        """
        # Return array of predictions (0 or 1)
        return (X[:,self.feature_idx]>=self.threshold).astype(np.uint8)


### Visual check: the learned threshold

The two histograms show the separate redshift bands; the dashed line is the cut the stump found all by itself.

In [ ]:
# ---------------------------------------------------------------------
# Visual check: the single redshift cut learned by the stump
# ---------------------------------------------------------------------
redshift = sample_df["redshift"].to_numpy()
is_star = (sample_df["class"] == "STAR").to_numpy()

stump_demo = DecisionStump()
X_1d = redshift.reshape(-1, 1)                 # a one-column feature matrix
y_1d = np.where(is_star, 0, 1)                 # 0 = STAR, 1 = non-STAR
stump_demo.fit(X_1d, y_1d)

plt.figure(figsize=(9, 4))
plt.hist(redshift[is_star], bins=80, alpha=0.6, label="STAR")
plt.hist(redshift[~is_star], bins=80, alpha=0.6, label="non-STAR")
plt.axvline(stump_demo.threshold, color="k", ls="--",
            label=f"learned threshold = {stump_demo.threshold:.3f}")
plt.xlabel("redshift")
plt.ylabel("count")
plt.title("One rule already separates STAR from non-STAR")
plt.legend()
plt.tight_layout()
plt.show()
print("accuracy of the single rule:", (stump_demo.predict(X_1d) == y_1d).mean())


## Section 3 — K-Nearest Neighbours

<center>
<img src="https://upload.wikimedia.org/wikipedia/commons/e/e7/KnnClassification.svg" width="400">
<br>
<em>(KNN classification logic. Source: Wikimedia Commons)</em>
</center>

A stump can only draw one straight cut; separating GALAXY from QSO needs a real multi-class model. **K-nearest neighbours** is the most intuitive one: to classify a point, find its $K$ nearest training neighbours and take a majority vote.

Distance is Euclidean,

$$d(x, y) = \sqrt{\sum_{i=1}^{D}(x_i - y_i)^2},$$

but we never take the square root — it is monotonic, so ordering by squared distance gives the same neighbours. Expanding the square lets us use a single matrix product:

$$\sum_{i=1}^{D}(x_i - y_i)^2 = \sum_{i=1}^{D}x_i^2 + \sum_{i=1}^{D}y_i^2 - 2\sum_{i=1}^{D}x_i y_i .$$

That is why the implementation pre-computes `X_train_square` and `X_test_square` and then only needs `X_batch @ X_train.T`. Two practical touches keep it usable: the distance matrix is built in **chunks** so an $(M \times N)$ array never blows up memory, and `np.argpartition` selects the $k$ smallest distances in linear time without fully sorting them.


In [ ]:
class KNN:
    # You can change the k value to see the prediction performance differences!
    def __init__(self, k=5):
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        """
        Lazy learning: just store the data.

        Args:
            X: Training features of shape (N, D).
            y: Training labels of shape (N,).
        """
        self.X_train = X
        self.y_train = y

    def predict(self, X_test: np.ndarray) -> np.ndarray:
      """
      Predicts labels for test data using chunking to avoid memory blow.
      """
      # 预计算训练样本的平方和 (N,)
      X_train_square = np.sum(np.square(self.X_train), axis=1)
      # 预计算测试样本的平方和 (M,)
      X_test_square = np.sum(np.square(X_test), axis=1)

      M = X_test.shape[0]
      chunk_size = 2048          # 根据你的RAM调整，2048通常安全
      predictions = []

      for start in range(0, M, chunk_size):
          end = min(start + chunk_size, M)
          X_batch = X_test[start:end]                # (batch_size, D)
          batch_square = X_test_square[start:end]    # (batch_size,)

          # 计算该批次与所有训练样本的距离平方矩阵 (batch_size, N)
          multiply = X_batch @ self.X_train.T        # (batch_size, N)
          dist_sq = batch_square[:, np.newaxis] + X_train_square[np.newaxis, :] - 2 * multiply

          # 找到每个测试样本最近的k个邻居的索引 (batch_size, k)
          nearest_indices = np.argpartition(dist_sq, kth=self.k-1, axis=-1)[:, :self.k]
          # 注意：argpartition 不保证完全排序，但前k个是k个最小的（无序）
          # 如果需要严格排序（多数投票不要求顺序），argpartition足够；若要按距离排序可用argsort但稍慢。

          # 根据邻居索引获取标签，并进行多数投票
          batch_preds = np.array([
              np.argmax(np.bincount(self.y_train[indices]))
              for indices in nearest_indices
          ])
          predictions.append(batch_preds)

      return np.concatenate(predictions)


### Visual check: decision regions

We project the high-dimensional problem onto two colour indices and colour the whole plane by what KNN predicts there.

In [ ]:
# ---------------------------------------------------------------------
# Visual check: KNN decision regions on a 2-D projection
# ---------------------------------------------------------------------
demo = sample_df.sample(3000, random_state=0)
colour = np.column_stack([demo["u"] - demo["g"], demo["g"] - demo["r"]])
colour = (colour - colour.min(0)) / (colour.max(0) - colour.min(0))
labels3 = demo["class"].map({"STAR": 0, "GALAXY": 1, "QSO": 2}).to_numpy()

n_fit = 2000
knn_demo = KNN(k=15)
knn_demo.fit(colour[:n_fit], labels3[:n_fit])

gx, gy = np.meshgrid(np.linspace(0, 1, 120), np.linspace(0, 1, 120))
grid = np.column_stack([gx.ravel(), gy.ravel()])
region = knn_demo.predict(grid).reshape(gx.shape)

plt.figure(figsize=(6.5, 5))
plt.contourf(gx, gy, region, levels=[-0.5, 0.5, 1.5, 2.5],
             colors=["#55A868", "#4C72B0", "#DD8452"], alpha=0.35)
plt.scatter(colour[n_fit:, 0], colour[n_fit:, 1], c=labels3[n_fit:],
            cmap="tab10", s=6, edgecolors="none")
plt.xlabel("u - g (normalised)")
plt.ylabel("g - r (normalised)")
plt.title("KNN (k=15) decision regions from two colour indices")
plt.tight_layout()
plt.show()


## Section 4 — K-Means Clustering

<center>
<img src="https://upload.wikimedia.org/wikipedia/commons/e/ea/K-means_convergence.gif" width="400">
<br>
<em>(K-Means iterative process. Source: Wikimedia Commons)</em>
</center>

Every model so far used the labels. **K-means** ignores them completely: it searches for $K$ natural groups in feature space, which lets us ask whether the data even *wants* to fall into three clusters.

$$\text{WCSS} = \sum_{i=1}^{N} \min_{j} \lVert x^{(i)} - \mu^{(j)} \rVert^2$$

The algorithm alternates two vectorised steps — assign every point to its nearest centroid, then move each centroid to the mean of the points assigned to it — until the centroids stop moving. Empty clusters simply keep their previous centroid.

To choose $K$ we use the **elbow method**: plot the within-cluster sum of squares against $K$ and pick the "elbow". Instead of eyeballing it, `find_optimal_k` picks the point with the largest perpendicular distance to the straight line joining the first and last $(K, \text{WCSS})$ points.


In [ ]:
class KMeans:
    def __init__(self, k=3, max_iters=100, seed=42):
        self.k = k
        self.max_iters = max_iters
        self.seed = seed
        self.centroids = None
        self.wcss = 0.0

    def fit(self, X: np.ndarray) -> Tuple[np.ndarray, float]:
        """
        Runs K-Means clustering.

        Args:
            X: Input data array of shape (N, D).

        Returns:
            clusters: Array of shape (N,) containing cluster indices.
            wcss: The Within-Cluster Sum of Squares.
        """
        np.random.seed(self.seed)
        N, D = X.shape
        # Initialize centroids (randomly)
        self.centroids=X[np.random.choice(N,self.k,replace=False)]
        centroid=self.centroids.copy()
        # Initialize clusters
        clusters = np.zeros(N, dtype=int)
        iter=1
        while iter<=self.max_iters:
        # Loop until convergence or max_iters:
            # 1. Assign clusters (vectorized distance)
            centroid=self.centroids.copy()
            C_sq=np.sum(np.square(self.centroids),axis=-1) #(self.k,)
            X_sq=np.sum(np.square(X),axis=-1) #(N,)
            multiply=X.dot(self.centroids.T)
            distance_squared=C_sq[np.newaxis,:]+X_sq[:,np.newaxis]-2*multiply
            if iter==1:
              clusters=np.argmin(distance_squared,axis=-1)
            else:
              old_clusters=clusters.copy()
              clusters=np.argmin(distance_squared,axis=-1)
            # 2. Update centroids (mean)
            one_hot=np.eye(self.k)[clusters]
            if iter==1:
              counts=np.sum(one_hot,axis=0)
            else:
              old_counts=counts.copy()
              counts=np.sum(one_hot,axis=0)
            sums=np.dot(one_hot.T,X)
            centroid[counts>0]=sums[counts>0]/counts[counts>0,np.newaxis]
            # 3. Check for convergence
            if np.allclose(centroid, self.centroids):
              break
            else:
              iter+=1
              self.centroids=centroid.copy()
        # Calculate final WCSS
        self.wcss=np.sum(np.square(np.abs(X-self.centroids[clusters]))) # Calculate real WCSS here

        return clusters, self.wcss


def find_optimal_k(X: np.ndarray, k_range: range) -> Tuple[int, np.ndarray]:
    """
    Finds optimal K by scanning a range and calculating the distance.
    You can use loop here, we don't detect in this function.
    You can assume that the len(k_range) >= 3 to make this function meaningful

    Args:
        X: Input data.
        k_range: List/Range of K values to test.

    Returns:
        optimal_k: The estimated optimal K.
        wcss_history: List of WCSS values for each K.
    """
    wcss_history = []

    # 1. Iterate over k_range
    for k in k_range:
    # 2. Fit KMeans and collect WCSS
      kmeans=KMeans(k=k)
      cluster,wcss=kmeans.fit(X)
      wcss_history.append(wcss)
    # 3. Calculate the distance of each point
    wcss_history=np.array(wcss_history)
    k_array=np.array(list(k_range))
    x1,y1=k_range[0],wcss_history[0]
    x2,y2=k_range[-1],wcss_history[-1]
    if (x2-x1)**2+(y2-y1)**2!=0:
      geometric_distance=np.abs((x2-x1)*(y1-wcss_history)-(x1-k_array)*(y2-y1))/np.sqrt((x2-x1)**2+(y2-y1)**2)
      optimal_k = k_range[np.argmax(geometric_distance)]
    else:
      return k_array[0],wcss_history

    return optimal_k, wcss_history


def plot_WCSS(optimal_k: int, wcss_history: np.ndarray) -> None:
    """
    Plots WCSS vs K and highlights the selected optimal K.

    Args:
        optimal_k: The optimal K returned by find_optimal_k.
        wcss_history: WCSS values ordered by K=1..len(wcss_history).
    """
    # 1. Build x-axis K values from wcss_history length
    xaxis=list(range(1,len(wcss_history)+1))
    # 2. Plot WCSS curve with marker='o'
    plt.plot(xaxis,wcss_history,marker='o')
    # 3. Set title/xlabel/ylabel and enable grid
    plt.title('Elbow Method: WCSS vs K')
    plt.xlabel('Number of Clusters (K)')
    plt.ylabel('WCSS')
    plt.grid()
    # 4. Highlight optimal_k with a vertical line or distinct marker
    plt.plot([optimal_k],[wcss_history[optimal_k-1]],marker='x')


### Visual check: clusters and the elbow

The left panel shows the clusters and their centroids; the right panel shows how the WCSS falls as K grows and where the elbow method stops.

In [ ]:
# ---------------------------------------------------------------------
# Visual check: K-means clusters and the elbow curve
# ---------------------------------------------------------------------
demo = sample_df.sample(5000, random_state=1)
colour = np.column_stack([demo["u"] - demo["g"], demo["g"] - demo["r"]])
colour = (colour - colour.min(0)) / (colour.max(0) - colour.min(0))

km_demo = KMeans(k=3)
clusters, wcss = km_demo.fit(colour)

k_values = range(1, 8)
optimal_k, wcss_history = find_optimal_k(colour, k_values)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(colour[:, 0], colour[:, 1], c=clusters, cmap="viridis", s=6)
axes[0].scatter(km_demo.centroids[:, 0], km_demo.centroids[:, 1],
                marker="X", s=180, c="red", edgecolors="k", label="centroids")
axes[0].set_title(f"K-means (k=3)  |  WCSS = {wcss:.3f}")
axes[0].set_xlabel("u - g")
axes[0].set_ylabel("g - r")
axes[0].legend()

axes[1].plot(list(k_values), wcss_history, marker="o")
axes[1].axvline(optimal_k, color="red", ls="--", label=f"elbow -> k = {optimal_k}")
axes[1].set_title("Elbow method")
axes[1].set_xlabel("k")
axes[1].set_ylabel("WCSS")
axes[1].legend()

plt.tight_layout()
plt.show()


## Section 5 — Per-Class Evaluation Metrics

<center>
<svg width="520" height="240" viewBox="0 0 520 240" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Confusion matrix layout">
  <rect x="0" y="0" width="520" height="240" fill="white"/>
  <text x="280" y="24" font-size="16" text-anchor="middle" fill="#2F3B52">Predicted label</text>
  <text x="28" y="140" font-size="16" text-anchor="middle" fill="#2F3B52" transform="rotate(-90 28 140)">True label</text>
  <rect x="140" y="50" width="280" height="160" fill="none" stroke="#2F3B52" stroke-width="2"/>
  <line x1="280" y1="50" x2="280" y2="210" stroke="#2F3B52" stroke-width="2"/>
  <line x1="140" y1="130" x2="420" y2="130" stroke="#2F3B52" stroke-width="2"/>
  <text x="210" y="92" font-size="18" text-anchor="middle" fill="#4C78A8">TP</text>
  <text x="350" y="92" font-size="18" text-anchor="middle" fill="#F58518">FP</text>
  <text x="210" y="172" font-size="18" text-anchor="middle" fill="#F58518">FN</text>
  <text x="350" y="172" font-size="18" text-anchor="middle" fill="#4C78A8">TN</text>
  <text x="210" y="44" font-size="13" text-anchor="middle" fill="#2F3B52">pred=1</text>
  <text x="350" y="44" font-size="13" text-anchor="middle" fill="#2F3B52">pred=0</text>
  <text x="132" y="92" font-size="13" text-anchor="end" fill="#2F3B52">true=1</text>
  <text x="132" y="172" font-size="13" text-anchor="end" fill="#2F3B52">true=0</text>
</svg>
<br>
<em>(Confusion Matrix: The basis for Precision and Recall)</em>
</center>

When one class holds 65 % of the data, accuracy is a misleading score, so we look at each class separately. For a class $C$:

$$\text{Precision}_C = \frac{TP}{TP + FP + \epsilon}, \qquad \text{Recall}_C = \frac{TP}{TP + FN + \epsilon}, \qquad \text{F1}_C = 2\,\frac{\text{Precision}_C \cdot \text{Recall}_C}{\text{Precision}_C + \text{Recall}_C + \epsilon}$$

The whole confusion matrix comes from a single `np.bincount` on the flattened index `true * C + pred`, with no Python loop over classes. Recall is exactly the building block of the balanced-accuracy metric from the introduction, which is why we compute these numbers by hand instead of importing them.


In [ ]:
def evaluate_per_class(y_true: np.ndarray, y_pred: np.ndarray, class_names: List[str]) -> pd.DataFrame:
    """
    Computes Precision, Recall, and F1 for each class.

    Args:
        y_true: True labels of shape (N,).
        y_pred: Predicted labels of shape (N,).
        class_names: List of class names (e.g., ['STAR', 'GALAXY', 'QSO']).

    Returns:
        metrics_df: A DataFrame where Index=Class and Columns=[Precision, Recall, F1].
    """
    metrics = []
    epsilon = 1e-9
    num_classes = len(class_names)

    # Vectorized Metric Calculation (No explicit class loop needed for logic, mostly)

    # 1. Compute Confusion Matrix (C x C)
    # Rows=True, Cols=Pred
    # Trick: Use bincount on flattened index (true * C + pred)
    flatten_idx=y_true*num_classes+y_pred
    confusion_matrix=np.bincount(flatten_idx,minlength=num_classes**2).reshape((num_classes,num_classes))
    print(confusion_matrix)
    # 2. Extract TP, FP, FN
    TP=confusion_matrix[np.arange(num_classes),np.arange(num_classes)]
    FP=np.sum(confusion_matrix,axis=0)-TP
    FN=np.sum(confusion_matrix,axis=1)-TP
    # 3. Compute Metrics
    Precision=TP/(TP+FP+epsilon)
    Recall=TP/(TP+FN+epsilon)
    F1=2*(Precision*Recall)/(Precision+Recall+epsilon)
    # 4. Format Output
    metrics=np.stack((Precision,Recall,F1),axis=-1)

    return pd.DataFrame(metrics, index=class_names, columns=['Precision', 'Recall', 'F1'])


### Visual check: the confusion matrix

The diagonal is what the model gets right; the off-diagonal cells are exactly the mistakes that drag the per-class recall down.

In [ ]:
# ---------------------------------------------------------------------
# Visual check: confusion matrix and per-class metrics
# ---------------------------------------------------------------------
demo = sample_df.sample(8000, random_state=2)
X_demo, y_demo, _ = preprocess_data(demo)
n_fit = 6000
knn_metrics = KNN(k=15)
knn_metrics.fit(X_demo[:n_fit], y_demo[:n_fit])
y_hat = knn_metrics.predict(X_demo[n_fit:])

names = ["STAR", "GALAXY", "QSO"]
metrics = evaluate_per_class(y_demo[n_fit:], y_hat, names)
cm = np.bincount(y_demo[n_fit:] * 3 + y_hat, minlength=9).reshape(3, 3)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks(range(3), names)
axes[0].set_yticks(range(3), names)
axes[0].set_xlabel("predicted")
axes[0].set_ylabel("true")
axes[0].set_title("Confusion matrix (KNN)")
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, ax=axes[0], shrink=0.8)

metrics.plot(kind="bar", ax=axes[1], rot=0, ylim=(0, 1))
axes[1].set_title("Per-class precision / recall / F1")
plt.tight_layout()
plt.show()


## Putting the classic models together

Now let us run the whole from-scratch toolbox on the real competition data: load `train.csv`, preprocess it, look at the redshift distribution, fit the decision stump, train a KNN classifier and print the per-class metrics.

Two practical notes:

* The hand-written KNN compares every test point with every training point, so fitting it on all 577k rows would need far too much memory. We therefore fit it on a random subset of the training data and score it on a held-out part of that subset. `test.csv` has no labels, so it cannot be used for scoring.
* This baseline exists to be *understood*; the models that actually compete begin in Section 6.


In [ ]:
# =====================================================================
# Classic baseline on the competition data
# =====================================================================
train_path = find_csv("train.csv")
train_df = pd.read_csv(train_path)
print("train:", train_df.shape)

# Statistics are computed on the training frame only (no data leakage).
X_train, y_train, _ = preprocess_data(train_df)

# --- Look at the redshift distribution --------------------------------
plot_redshift_distribution(train_df)
plt.show()

# --- Section 2: the decision stump, on the full training set ----------
print("\n--- Decision stump (STAR vs non-STAR) ---")
stump = DecisionStump()
stump.fit(X_train, y_train)
print(f"optimal redshift threshold: {stump.threshold:.4f}")
print(f"stump accuracy on train: {(stump.predict(X_train) == (y_train != 0).astype(int)).mean():.4f}")

# --- Section 3 & 5: KNN on a subset -----------------------------------
# The from-scratch KNN is O(N_test * N_train); on the full data that is far
# too heavy, so we fit it on a random subset and score it on a held-out part.
rng = np.random.default_rng(42)
idx = rng.permutation(len(X_train))[:20000]
X_sub, y_sub = X_train[idx], y_train[idx]
n_fit = 16000

print("\n--- K-nearest neighbours (k = 100) ---")
knn = KNN(k=100)
knn.fit(X_sub[:n_fit], y_sub[:n_fit])
y_pred = knn.predict(X_sub[n_fit:])
print(f"KNN accuracy on the held-out subset: {(y_pred == y_sub[n_fit:]).mean():.4f}")

print("\n--- Per-class metrics (KNN) ---")
display(evaluate_per_class(y_sub[n_fit:], y_pred, ["STAR", "GALAXY", "QSO"]))

# --- Section 4: the K-means elbow, on a subset ------------------------
print("\n--- K-means: searching for the optimal K ---")
sub = X_train[rng.permutation(len(X_train))[:5000]]
optimal_k, wcss_history = find_optimal_k(sub, range(1, 7))
plot_WCSS(optimal_k, wcss_history)
plt.show()
print(f"optimal K detected by the elbow method: {optimal_k}")


## Deep Models (Sections 6–14)

Sections 1–5 built the classic algorithms by hand, on a deliberately small feature set, so that the maths stays readable. From here on we take the training wheels off: we are allowed `scikit-learn` and `PyTorch`, and we optimise the real competition metric.

| | Sections 1–5 | Sections 6–14 |
| :--- | :--- | :--- |
| **Goal** | understand the algorithms | maximise the leaderboard score |
| **Libraries** | `numpy` / `pandas` only | `scikit-learn`, `PyTorch`, … |
| **Features** | 8, to keep the maths readable | 14, engineered for accuracy |

Section 6 sets up everything the later sections share, Section 7 establishes a strong classical baseline, Sections 8–12 introduce the MLP, RNN, LSTM, Transformer and GAN, and Section 13 blends them all into the final `submission.csv`.


## Section 6 — Competition Setup: Metric, Split, and Shared Utilities

### 6.1 Leakage-free validation

We cannot tune models on the test set (we do not have its labels). So we **carve a validation set out of the training data** with a *stratified* split, so that every class keeps its proportion of 65 / 20 / 14 % in both halves.

We also obey the same leakage rule as in Section 1: **all normalisation statistics (min / max) are computed on the training half only** and then applied to the validation and test halves. Computing them on the full training set would leak validation information into the model — a silent, easy-to-make mistake.

### 6.2 Feature engineering — colour indices

The five magnitudes `u, g, r, i, z` measure how bright an object is in five different colour filters. But an object's *absolute* brightness depends on how far away it is. What really classifies a star versus a galaxy versus a quasar is its **colour**, i.e. the *difference* between two adjacent bands:

$$u-g,\qquad g-r,\qquad r-i,\qquad i-z$$

These four **colour indices** are the coordinates of the famous *colour–colour diagram* that astronomers actually use. Feeding them to a model is a classic example of *domain knowledge as feature engineering* — cheap to compute, and often worth more than a fancier model.

We also keep the two categorical columns (`spectral_type`, `galaxy_population`) as small integer codes. We end up with **14 features**: `alpha, delta, u, g, r, i, z, redshift` + 2 categorical codes + 4 colour indices.

### 6.3 Class weights

To fight the imbalance we give each class a weight inversely proportional to its frequency:

$$w_c = \frac{N}{C \cdot n_c}$$

so that a mistake on a rare STAR costs about $2.32\times$ more than a mistake on a common GALAXY. These weights go straight into the loss function (Sections 8–11).

### 6.4 Shared training utilities

Every neural network in this part is trained by the *same* loop, so we write it once:

* `train_torch(...)` — mini-batch loop with **AdamW**, a **One-Cycle learning-rate schedule**, class-weighted **cross-entropy**, and **best-checkpoint selection by balanced accuracy**;
* `predict_proba(...)` — batched softmax probabilities (needed later for the ensemble);
* `balanced_accuracy(...)` / `report(...)` — our own metric helpers.

Two knobs at the top (`NN_TRAIN_SIZE`, `*_EPOCHS`) let you trade time for accuracy. The defaults are sized for a free Colab GPU.


In [ ]:
# =====================================================================
# Section 6 - Competition setup
# =====================================================================
import os
import glob
import copy
import time
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if DEVICE.type == "cpu":
    print("WARNING: no GPU found. In Colab use Runtime > Change runtime type > GPU (T4).")

# --- Locate the CSVs (find_csv is defined in the setup cell) -----------
train_path, test_path = find_csv("train.csv"), find_csv("test.csv")
DATA_DIR = os.path.dirname(train_path)
print("DATA_DIR:", DATA_DIR or ".")

# --- Feature definition -------------------------------------------------
CLASS_MAP = {"STAR": 0, "GALAXY": 1, "QSO": 2}
INV_CLASS_MAP = {v: k for k, v in CLASS_MAP.items()}
SPECTRAL_MAP = {"M": 0, "A/F": 1, "O/B": 2, "G/K": 3}
POPULATION_MAP = {"Red_Sequence": 0, "Blue_Cloud": 1}
RAW_FEATURES = ["alpha", "delta", "u", "g", "r", "i", "z", "redshift"]
CATEGORICAL_FEATURES = ["spectral_type_code", "galaxy_population_code"]
COLOR_INDEX_FEATURES = ["u_g", "g_r", "r_i", "i_z"]
FEATURES = RAW_FEATURES + CATEGORICAL_FEATURES + COLOR_INDEX_FEATURES

def add_engineered_features(df):
    df = df.copy()
    df["spectral_type_code"] = df["spectral_type"].map(SPECTRAL_MAP).astype("float64")
    df["galaxy_population_code"] = df["galaxy_population"].map(POPULATION_MAP).astype("float64")
    df["u_g"] = df["u"] - df["g"]
    df["g_r"] = df["g"] - df["r"]
    df["r_i"] = df["r"] - df["i"]
    df["i_z"] = df["i"] - df["z"]
    return df

def build_features(df, feature_cols, stats=None):
    """Encode, engineer and min-max normalise. Stats must come from the training split only."""
    df = add_engineered_features(df)
    if stats is None:
        stats = {c: {"min": float(df[c].min()), "max": float(df[c].max())} for c in feature_cols}
    for c in feature_cols:
        span = stats[c]["max"] - stats[c]["min"]
        df[c] = (df[c] - stats[c]["min"]) / span if span != 0 else 0.0
    X = df[feature_cols].to_numpy(dtype="float32")
    y = df["class"].map(CLASS_MAP).to_numpy() if "class" in df.columns else None
    return X, y, stats

# --- Load and split -----------------------------------------------------
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
print("train:", train_df.shape, " test:", test_df.shape)

tr_df, va_df = train_test_split(train_df, test_size=0.2, random_state=SEED, stratify=train_df["class"])
X_tr, y_tr, STATS = build_features(tr_df, FEATURES)
X_va, y_va, _ = build_features(va_df, FEATURES, stats=STATS)
X_te, _, _ = build_features(test_df, FEATURES, stats=STATS)
print("X_tr", X_tr.shape, "| X_va", X_va.shape, "| X_te", X_te.shape)

# --- Class weights for the imbalanced loss ------------------------------
counts = np.bincount(y_tr, minlength=3)
class_weights = torch.tensor(len(y_tr) / (3.0 * counts), dtype=torch.float32, device=DEVICE)
for i, name in enumerate(["STAR", "GALAXY", "QSO"]):
    print(f"  {name:7s} n={counts[i]:7d}  weight={class_weights[i]:.3f}")

# --- Metric helpers -----------------------------------------------------
def balanced_accuracy(y_true, y_pred, n_classes=3):
    recalls = [float((y_pred[y_true == c] == c).mean()) if (y_true == c).any() else 0.0
               for c in range(n_classes)]
    return float(np.mean(recalls))

def report(name, y_true, y_pred):
    print(f"{name:14s} accuracy={accuracy_score(y_true, y_pred):.4f}   "
          f"balanced_acc={balanced_accuracy(y_true, y_pred):.4f}   "
          f"macro_f1={f1_score(y_true, y_pred, average='macro'):.4f}")

@torch.no_grad()
def predict_proba(model, X, batch_size=8192):
    """Batched softmax probabilities for any torch classifier."""
    model.eval()
    probs = []
    for i in range(0, len(X), batch_size):
        xb = torch.from_numpy(np.ascontiguousarray(X[i:i + batch_size])).to(DEVICE)
        probs.append(torch.softmax(model(xb), dim=-1).cpu())
    return torch.cat(probs).numpy()

# --- The shared training loop ------------------------------------------
def train_torch(model, X_tr_, y_tr_, X_va_, y_va_, epochs=25, batch_size=1024, lr=2e-3,
                weight_decay=1e-4, label_smoothing=0.0, verbose=True):
    model.to(DEVICE)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(np.ascontiguousarray(X_tr_)),
                      torch.from_numpy(np.asarray(y_tr_)).long()),
        batch_size=batch_size, shuffle=True, drop_last=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, total_steps=epochs * max(1, len(loader)), pct_start=0.3)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)

    best_score, best_state = -1.0, None
    history = {"loss": [], "val_bal": []}
    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            if scheduler.last_epoch < scheduler.total_steps - 1:
                scheduler.step()
            running += loss.item() * len(xb)
        epoch_loss = running / len(loader.dataset)
        pred = predict_proba(model, X_va_).argmax(1)
        score = balanced_accuracy(y_va_, pred)
        history["loss"].append(epoch_loss)
        history["val_bal"].append(score)
        if score > best_score:
            best_score, best_state = score, copy.deepcopy(model.state_dict())
        if verbose and (epoch % 5 == 0 or epoch == epochs):
            print(f"  epoch {epoch:02d}  loss={epoch_loss:.4f}  "
                  f"val_balanced_acc={score:.4f}")
    if best_state is not None:
        model.load_state_dict(best_state)
    model.history_ = history
    print(f"  -> best val_balanced_acc = {best_score:.4f}")
    return model

# --- Plotting helper shared by the deep-model sections ------------------
def plot_history(model, title):
    """Per-epoch training loss (left axis) and validation balanced accuracy (right axis)."""
    h = model.history_
    epochs = np.arange(1, len(h["loss"]) + 1)
    fig, ax1 = plt.subplots(figsize=(8, 3.5))
    ax1.plot(epochs, h["loss"], color="#4C72B0", marker="o", ms=3)
    ax1.set_xlabel("epoch")
    ax1.set_ylabel("train loss", color="#4C72B0")
    ax1.tick_params(axis="y", labelcolor="#4C72B0")
    ax1.grid(alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(epochs, h["val_bal"], color="#DD8452", marker="s", ms=3)
    ax2.set_ylabel("validation balanced accuracy", color="#DD8452")
    ax2.tick_params(axis="y", labelcolor="#DD8452")

    best = int(np.argmax(h["val_bal"]))
    ax2.axvline(epochs[best], color="grey", ls=":", label=f"best epoch = {epochs[best]}")
    ax2.legend(loc="center right", fontsize=8)
    ax1.set_title(title)
    fig.tight_layout()
    plt.show()

# --- Knobs (lower them if you are in a hurry / on CPU) ------------------
NN_TRAIN_SIZE = 200_000
MLP_EPOCHS = 30
SEQ_EPOCHS = 25
GAN_STEPS = 2000
print("\nSetup ready.")


### Visual check: correlations and loss weights

The colour indices are strongly correlated with each other, and the loss weight is largest for the rarest class.

In [ ]:
# ---------------------------------------------------------------------
# Visual check: feature correlations and the loss weights
# ---------------------------------------------------------------------
idx = np.random.choice(len(X_tr), size=min(20000, len(X_tr)), replace=False)
corr = pd.DataFrame(X_tr[idx], columns=FEATURES).corr()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
im = axes[0].imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
axes[0].set_xticks(range(len(FEATURES)))
axes[0].set_xticklabels(FEATURES, rotation=90, fontsize=7)
axes[0].set_yticks(range(len(FEATURES)))
axes[0].set_yticklabels(FEATURES, fontsize=7)
axes[0].set_title("Feature correlation (14 engineered features)")
fig.colorbar(im, ax=axes[0], shrink=0.8)

weights = class_weights.detach().cpu().numpy()
axes[1].bar(["STAR", "GALAXY", "QSO"], weights, color=["#55A868", "#4C72B0", "#DD8452"])
for x, v in enumerate(weights):
    axes[1].text(x, v, f"{v:.2f}", ha="center", va="bottom")
axes[1].set_title("Class weights in the loss (rare class -> larger weight)")
plt.tight_layout()
plt.show()


## Section 7 — A Strong Classical Baseline: Gradient-Boosted Trees

Before reaching for neural networks, it is a professional habit to establish a **strong baseline**. For *tabular* data (rows × columns, no images, no audio) the strongest classical family is **gradient-boosted decision trees**.

### The idea in three steps

1. **A decision tree** splits the data with questions like "*is redshift < 0.0016?*". A single tree is a weak, high-variance model.
2. **Boosting** turns weakness into strength: fit a tree, look at the examples it got wrong, then fit the *next* tree on those mistakes. Each new tree corrects the previous ensemble. After a few hundred rounds you have a very accurate model.
3. **Histogram-based** training (what `HistGradientBoostingClassifier` does) bins each feature into ≤ 255 buckets, which makes it blazing fast and memory-light on 500k rows.

### Why `class_weight="balanced"`?

Remember the metric: *balanced* accuracy. Thanks to the class weights, the trees are penalised more for missing a STAR, so the model leans toward predicting rare classes — exactly what the metric rewards. You will see accuracy go **down** slightly while balanced accuracy goes **up**: that is the metric doing its job.

### Reading the output

* The printed table compares `accuracy`, `balanced_acc` and `macro_f1`.
* `permutation_importance` shuffles one feature at a time and measures how much the score drops. This tells us *which physical quantity matters* — you should see `redshift` at the top, followed by the colour indices, which is a nice sanity check against the astronomy from Section 2.


In [ ]:
# =====================================================================
# Section 7 - Gradient-boosted trees baseline
# =====================================================================
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance

t0 = time.time()
gbdt = HistGradientBoostingClassifier(
    max_iter=400,
    learning_rate=0.08,
    max_leaf_nodes=63,
    class_weight="balanced",   # optimise the metric that Kaggle actually uses
    random_state=SEED,
)
gbdt.fit(X_tr, y_tr)
proba_gbdt = gbdt.predict_proba(X_va)
report("GBDT baseline", y_va, proba_gbdt.argmax(1))
print(f"training time: {time.time() - t0:.1f}s")

# --- Which features carry the signal? -----------------------------------
sub = np.random.choice(len(X_va), size=min(20000, len(X_va)), replace=False)
perm = permutation_importance(gbdt, X_va[sub], y_va[sub], n_repeats=3,
                              random_state=SEED, scoring="balanced_accuracy", n_jobs=-1)
imp = (pd.DataFrame({"feature": FEATURES, "importance": perm.importances_mean})
         .sort_values("importance", ascending=False).reset_index(drop=True))
display(imp)

plt.figure(figsize=(8, 5))
plt.barh(imp["feature"][::-1], imp["importance"][::-1])
plt.xlabel("Drop in balanced accuracy when the feature is shuffled")
plt.title("Permutation Importance (Gradient-Boosted Trees)")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


### Visual check: where the trees get confused

The row-normalised matrix is literally the per-class recall sitting on its diagonal.

In [ ]:
# ---------------------------------------------------------------------
# Visual check: where the gradient-boosted trees get confused
# ---------------------------------------------------------------------
names = ["STAR", "GALAXY", "QSO"]
cm = confusion_matrix(y_va, proba_gbdt.argmax(1))
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(cm, cmap="Greens")
axes[0].set_xticks(range(3), names)
axes[0].set_yticks(range(3), names)
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
axes[0].set_xlabel("predicted")
axes[0].set_ylabel("true")
axes[0].set_title("GBDT confusion matrix (counts)")

axes[1].imshow(cm_norm, cmap="Greens", vmin=0, vmax=1)
axes[1].set_xticks(range(3), names)
axes[1].set_yticks(range(3), names)
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", color="black")
axes[1].set_xlabel("predicted")
axes[1].set_ylabel("true")
axes[1].set_title("Row-normalised (the diagonal is per-class recall)")
plt.tight_layout()
plt.show()


## Section 8 — Multi-Layer Perceptron (MLP)

Now the deep learning begins. The MLP is the "hello world" of neural networks and, on tabular data, also the strongest of the five architectures we will meet.

### 8.1 One neuron

A single artificial neuron computes an **affine transformation followed by a non-linearity**:

$$z = \mathbf{w}^\top \mathbf{x} + b, \qquad a = \sigma(z)$$

* $\mathbf{w}$ are the **weights** (learned), $b$ the **bias** (learned);
* $\sigma$ is an **activation function**. Without it, stacking layers would collapse into one big linear map — and a linear model cannot separate galaxies from quasars.

### 8.2 From neuron to network

An MLP stacks neurons into **layers**. Layer $\ell$ takes the previous layer's activations and produces new ones:

$$\mathbf{a}^{(\ell)} = \sigma\!\left(\mathbf{W}^{(\ell)}\mathbf{a}^{(\ell-1)} + \mathbf{b}^{(\ell)}\right)$$

The last layer outputs **3 numbers (logits)**, one per class. A **softmax** turns them into probabilities:

$$\hat{p}_c = \frac{e^{z_c}}{\sum_{j} e^{z_j}}$$

### 8.3 How it learns — back-propagation + gradient descent

1. **Forward pass:** push a mini-batch of rows through the network, get probabilities.
2. **Loss:** compare with the truth using **cross-entropy** (with our per-class weights $w_c$):

$$\mathcal{L} = -\frac{1}{B}\sum_{i=1}^{B} w_{y_i}\,\log \hat{p}_{i,\,y_i}$$

3. **Backward pass:** autograd applies the chain rule and gives $\partial\mathcal{L}/\partial\theta$ for every parameter $\theta$.
4. **Update:** nudge each parameter down the gradient, $\theta \leftarrow \theta - \eta\, \nabla_\theta\mathcal{L}$, and repeat.

This is exactly the loop inside `train_torch`.

### 8.4 Our architecture

```
input(14) -> Linear(256) -> BatchNorm -> ReLU -> Dropout
          -> Linear(256) -> BatchNorm -> ReLU -> Dropout
          -> Linear(128) -> BatchNorm -> ReLU -> Dropout
          -> Linear(3)
```

* **BatchNorm** re-centres each layer's inputs, which stabilises and speeds up training.
* **Dropout** randomly zeroes 20 % of activations during training, forcing the network not to rely on any single unit — a cheap, effective regulariser.
* **AdamW** is a gradient-descent variant with per-parameter adaptive step sizes plus decoupled weight decay — the default choice of practitioners.
* **One-Cycle LR**: the learning rate ramps up then decays; it trains fast and lands in a good minimum.

> **Key idea to remember:** everything above is *learned*. We never told it that redshift matters — it discovered that by itself.


In [ ]:
# =====================================================================
# Section 8 - Multi-Layer Perceptron
# =====================================================================
class MLP(nn.Module):
    """A plain feed-forward network: Linear -> BatchNorm -> ReLU -> Dropout, stacked."""
    def __init__(self, n_features, hidden=(256, 256, 128), n_classes=3, dropout=0.2):
        super().__init__()
        layers, prev = [], n_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# Subsample the training set so that one epoch stays cheap; raise NN_TRAIN_SIZE for more accuracy.
idx = np.random.choice(len(X_tr), size=min(NN_TRAIN_SIZE, len(X_tr)), replace=False)
X_nn, y_nn = X_tr[idx], y_tr[idx]
print("neural-net training set:", X_nn.shape)

t0 = time.time()
mlp = MLP(X_nn.shape[1])
mlp = train_torch(mlp, X_nn, y_nn, X_va, y_va, epochs=MLP_EPOCHS, batch_size=4096, lr=3e-3)
proba_mlp = predict_proba(mlp, X_va)
report("MLP", y_va, proba_mlp.argmax(1))
print(f"training time: {time.time() - t0:.1f}s")


### Visual check: the MLP learning curve

Watch the loss fall and the validation balanced accuracy rise; the dotted line marks the best epoch, whose weights are the ones kept.

In [ ]:
# ---------------------------------------------------------------------
# Visual check: the MLP learning curve and its mistakes
# ---------------------------------------------------------------------
plot_history(mlp, "MLP training curve")

names = ["STAR", "GALAXY", "QSO"]
cm = confusion_matrix(y_va, proba_mlp.argmax(1))
plt.figure(figsize=(4.5, 4))
plt.imshow(cm, cmap="Purples")
plt.xticks(range(3), names)
plt.yticks(range(3), names)
for i in range(3):
    for j in range(3):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.xlabel("predicted")
plt.ylabel("true")
plt.title("MLP confusion matrix")
plt.tight_layout()
plt.show()


## Section 9 — Recurrent Neural Networks (RNN)

### 9.1 When do we use an RNN?

RNNs were designed for **sequences**: words in a sentence, frames in a video, prices over time. The defining feature is a **hidden state** $\mathbf{h}_t$ that is carried from step to step, acting as a running memory of everything seen so far.

Our data is a table, not a sentence — but we can still *view* it as a sequence: treat each of the 14 features as one **token** and read them one by one. This is mostly a pedagogical framing, and it is exactly how "tabular transformers" and "feature-token RNNs" are built in practice.

### 9.2 The recurrence

At step $t$ the cell receives the token $\mathbf{x}_t$ (here a scalar projected to a vector) and the previous hidden state:

$$\mathbf{h}_t = \tanh\!\left(\mathbf{W}_{x}\mathbf{x}_t + \mathbf{W}_{h}\,\mathbf{h}_{t-1} + \mathbf{b}\right)$$

* $\mathbf{W}_x$ mixes the **new input**, $\mathbf{W}_h$ mixes the **memory**;
* the **same** weights are reused at every step — this weight sharing is what makes an RNN handle variable-length input with a fixed number of parameters;
* after the last token, $\mathbf{h}_{T}$ is a summary of the whole row and is fed to a small classifier head.

### 9.3 The vanishing-gradient problem

Training an RNN means back-propagating *through time*:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{W}_h} \sim \prod_{t} \mathbf{W}_h^\top \,\mathrm{diag}\!\left(1-\mathbf{h}_t^2\right)$$

If the factors are smaller than 1, the product shrinks exponentially with the sequence length — the gradient **vanishes**, and the network forgets the early tokens. If they are larger than 1, it explodes. Section 10 solves the first problem with gates.

### 9.4 In code

Because we want you to *see* the recurrence, we write the loop by hand instead of calling `nn.RNN`:

```python
h = torch.zeros(batch, hidden)
for t in range(n_features):
    h = torch.tanh(Wx(tokens[:, t]) + Wh(h))
```

That `for` loop **is** the RNN. Everything else (embedding of the scalar feature, head, training) is plumbing.


In [ ]:
# =====================================================================
# Section 9 - Recurrent Neural Network (hand-written recurrence)
# =====================================================================
class TabularRNN(nn.Module):
    """Reads the D features one at a time and keeps a single hidden state."""
    def __init__(self, n_features, d_model=64, hidden=128, n_classes=3, dropout=0.2):
        super().__init__()
        self.n_features = n_features
        self.embed = nn.Linear(1, d_model)          # scalar feature -> vector token
        self.wx = nn.Linear(d_model, hidden)        # input -> hidden
        self.wh = nn.Linear(hidden, hidden, bias=False)  # hidden -> hidden
        self.head = nn.Sequential(nn.LayerNorm(hidden), nn.Dropout(dropout),
                                  nn.Linear(hidden, n_classes))

    def forward(self, x):
        h = torch.zeros(x.size(0), self.wh.out_features, device=x.device)
        tokens = self.embed(x.unsqueeze(-1))        # (B, D, d_model)
        for t in range(self.n_features):
            h = torch.tanh(self.wx(tokens[:, t]) + self.wh(h))
        return self.head(h)

t0 = time.time()
rnn = TabularRNN(X_nn.shape[1])
rnn = train_torch(rnn, X_nn, y_nn, X_va, y_va, epochs=SEQ_EPOCHS, batch_size=1024, lr=2e-3)
proba_rnn = predict_proba(rnn, X_va)
report("RNN", y_va, proba_rnn.argmax(1))
print(f"training time: {time.time() - t0:.1f}s")


### Visual check: the RNN learning curve

The same picture as the MLP, except the model now reads the features one step at a time.

In [ ]:
# ---------------------------------------------------------------------
# Visual check: the RNN learning curve
# ---------------------------------------------------------------------
plot_history(rnn, "RNN training curve")

names = ["STAR", "GALAXY", "QSO"]
cm = confusion_matrix(y_va, proba_rnn.argmax(1))
plt.figure(figsize=(4.5, 4))
plt.imshow(cm, cmap="Oranges")
plt.xticks(range(3), names)
plt.yticks(range(3), names)
for i in range(3):
    for j in range(3):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.xlabel("predicted")
plt.ylabel("true")
plt.title("RNN confusion matrix")
plt.tight_layout()
plt.show()


## Section 10 — Long Short-Term Memory (LSTM)

The LSTM (Hochreiter & Schmidhuber, 1997) fixes the vanishing gradient by replacing the plain hidden state with a **cell state** $\mathbf{c}_t$ that flows down a "conveyor belt", controlled by three **gates**.

### 10.1 The gates

A gate is a sigmoid layer producing values in $(0,1)$: 0 means "block completely", 1 means "let everything through".

$$
\begin{aligned}
\mathbf{i}_t &= \sigma\!\left(\mathbf{W}_i[\mathbf{x}_t,\mathbf{h}_{t-1}] + \mathbf{b}_i\right) && \text{input gate: how much new info to write}\\
\mathbf{f}_t &= \sigma\!\left(\mathbf{W}_f[\mathbf{x}_t,\mathbf{h}_{t-1}] + \mathbf{b}_f\right) && \text{forget gate: what to erase from memory}\\
\mathbf{o}_t &= \sigma\!\left(\mathbf{W}_o[\mathbf{x}_t,\mathbf{h}_{t-1}] + \mathbf{b}_o\right) && \text{output gate: what to expose}\\
\tilde{\mathbf{c}}_t &= \tanh\!\left(\mathbf{W}_c[\mathbf{x}_t,\mathbf{h}_{t-1}] + \mathbf{b}_c\right) && \text{candidate memory}
\end{aligned}
$$

Then the state update:

$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t, \qquad \mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t)$$

### 10.2 Why it helps

Look at the cell-state equation. The gradient path from $\mathbf{c}_t$ back to $\mathbf{c}_{t-1}$ is multiplied by $\mathbf{f}_t$ — a *learned* gate, not a fixed matrix. The network can therefore learn $\mathbf{f}_t \approx 1$ and keep information alive across many steps. That single additive path is the whole trick.

> **Implementing tip.** Instead of four separate `nn.Linear` layers we use **one** `nn.Linear(d_model, 4*hidden)` and then `chunk` the result into the four gates. Mathematically identical, but a single matrix–matrix product is faster on a GPU — a habit worth copying.


In [ ]:
# =====================================================================
# Section 10 - LSTM (hand-written gates)
# =====================================================================
class TabularLSTM(nn.Module):
    """Same sequence view as the RNN, but with an input/forget/output gated cell state."""
    def __init__(self, n_features, d_model=64, hidden=128, n_classes=3, dropout=0.2):
        super().__init__()
        self.n_features, self.hidden = n_features, hidden
        self.embed = nn.Linear(1, d_model)
        self.x2g = nn.Linear(d_model, 4 * hidden)       # input  -> 4 gates
        self.h2g = nn.Linear(hidden, 4 * hidden, bias=False)  # hidden -> 4 gates
        self.head = nn.Sequential(nn.LayerNorm(hidden), nn.Dropout(dropout),
                                  nn.Linear(hidden, n_classes))

    def forward(self, x):
        B = x.size(0)
        h = torch.zeros(B, self.hidden, device=x.device)
        c = torch.zeros(B, self.hidden, device=x.device)
        tokens = self.embed(x.unsqueeze(-1))
        for t in range(self.n_features):
            gates = self.x2g(tokens[:, t]) + self.h2g(h)          # (B, 4*hidden)
            i, f, g, o = gates.chunk(4, dim=-1)
            i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
            g = torch.tanh(g)
            c = f * c + i * g          # cell state: forget old + write new
            h = o * torch.tanh(c)      # hidden state: gated read-out
        return self.head(h)

t0 = time.time()
lstm = TabularLSTM(X_nn.shape[1])
lstm = train_torch(lstm, X_nn, y_nn, X_va, y_va, epochs=SEQ_EPOCHS, batch_size=1024, lr=2e-3)
proba_lstm = predict_proba(lstm, X_va)
report("LSTM", y_va, proba_lstm.argmax(1))
print(f"training time: {time.time() - t0:.1f}s")


### Visual check: gates over the sequence

Below the learning curve we watch the input / forget / output gates react to each feature step.

In [ ]:
# ---------------------------------------------------------------------
# Visual check: LSTM gates reacting to each feature step
# ---------------------------------------------------------------------
plot_history(lstm, "LSTM training curve")

lstm.eval()
with torch.no_grad():
    xb = torch.from_numpy(np.ascontiguousarray(X_va[:2048])).to(DEVICE)
    h = torch.zeros(xb.size(0), lstm.hidden, device=DEVICE)
    c = torch.zeros(xb.size(0), lstm.hidden, device=DEVICE)
    tokens = lstm.embed(xb.unsqueeze(-1))
    gates_over_time = []
    for t in range(lstm.n_features):
        gates = lstm.x2g(tokens[:, t]) + lstm.h2g(h)
        i, f, g, o = gates.chunk(4, dim=-1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c + i * g
        h = o * torch.tanh(c)
        gates_over_time.append([float(i.mean()), float(f.mean()),
                                float(o.mean()), float(g.abs().mean())])
gates_over_time = np.array(gates_over_time)

plt.figure(figsize=(9, 4))
for j, gate_name in enumerate(["input gate", "forget gate", "output gate", "|candidate|"]):
    plt.plot(range(len(FEATURES)), gates_over_time[:, j], marker="o", label=gate_name)
plt.xticks(range(len(FEATURES)), FEATURES, rotation=90, fontsize=7)
plt.xlabel("feature step (the sequence order)")
plt.ylabel("mean gate activation")
plt.title("LSTM: how the gates react to each feature")
plt.legend()
plt.tight_layout()
plt.show()


## Section 11 — The Transformer

"Attention Is All You Need" (Vaswani et al., 2017) replaced recurrence with **attention**: instead of passing a memory step by step, every token looks at every other token *in parallel* and decides what to pay attention to.

### 11.1 Scaled dot-product attention

Each token is projected into three vectors — a **Query** (what I am looking for), a **Key** (what I offer) and a **Value** (what I actually contain):

$$\mathbf{Q} = \mathbf{X}\mathbf{W}_Q,\qquad \mathbf{K} = \mathbf{X}\mathbf{W}_K,\qquad \mathbf{V} = \mathbf{X}\mathbf{W}_V$$

Attention is then

$$\mathrm{Attn}(\mathbf{Q},\mathbf{K},\mathbf{V}) = \mathrm{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}\right)\mathbf{V}$$

* $\mathbf{Q}\mathbf{K}^\top$ is a **similarity matrix** between every pair of tokens: how much token $i$ cares about token $j$;
* dividing by $\sqrt{d_k}$ keeps the logits from exploding as the head dimension grows (otherwise softmax saturates and gradients die);
* multiplying by $\mathbf{V}$ produces a **weighted average** of all values — this is the "attention" output.

### 11.2 Multi-head attention

One attention map can only capture one kind of relation. So we run $h$ attentions in parallel on low-dimensional projections ("heads") and concatenate them:

$$\mathrm{MHA}(\mathbf{X}) = [\mathrm{head}_1;\dots;\mathrm{head}_h]\mathbf{W}_O$$

This is why we `reshape` the (B, T, 3·d_model) projection into `(3, B, n_heads, T, d_head)` — a favourite interview-style reshape.

### 11.3 The encoder block

Each block is a standard sandwich:

$$\mathbf{X} \leftarrow \mathbf{X} + \mathrm{MHA}(\mathrm{LN}(\mathbf{X})), \qquad \mathbf{X} \leftarrow \mathbf{X} + \mathrm{FFN}(\mathrm{LN}(\mathbf{X}))$$

* **residual connections** ($\mathbf{X} + \dots$) give the gradient a shortcut straight to the bottom — essential for depth;
* **LayerNorm** stabilises the statistics of each token;
* the **FFN** is a per-token MLP (here `Linear → GELU → Linear`) that adds non-linear processing power.

### 11.4 Positional encoding

Attention is permutation-invariant: without extra information it cannot tell token 1 from token 14. We therefore add a **learnable positional embedding** to each token, so the model knows *which* feature it is looking at.

### 11.5 Reading out a class

A transformer returns a sequence. For classification we **mean-pool** the token vectors (an alternative is a dedicated `[CLS]` token) and apply `LayerNorm → Linear(3)`.

> **Honest note.** For 14 independent tabular columns, attention has far less structure to exploit than in language, so a Transformer will usually *not* beat gradient-boosted trees here. We include it because it is the architecture behind modern LLMs, and because diverse models make a better ensemble (Section 13).


In [ ]:
# =====================================================================
# Section 11 - Transformer encoder (hand-written attention)
# =====================================================================
class MultiHeadSelfAttention(nn.Module):
    """Scaled dot-product attention with several heads in parallel."""
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model, self.n_heads = d_model, n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, _ = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.d_head).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]                       # (B, heads, T, d_head)
        att = torch.softmax((q @ k.transpose(-2, -1)) / math.sqrt(self.d_head), dim=-1)
        out = (att @ v).transpose(1, 2).reshape(B, T, self.d_model)
        return self.proj(out)

class EncoderBlock(nn.Module):
    """Pre-LN transformer block: attention sub-layer + feed-forward sub-layer."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.drop(self.attn(self.norm1(x)))   # residual + attention
        x = x + self.drop(self.ff(self.norm2(x)))     # residual + MLP
        return x

class TabularTransformer(nn.Module):
    """Each feature is a token; mean-pool the encoder output and classify."""
    def __init__(self, n_features, d_model=64, n_heads=4, d_ff=128, n_layers=2,
                 n_classes=3, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.embed = nn.Linear(1, d_model)
        self.pos = nn.Parameter(torch.zeros(1, n_features, d_model))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.blocks = nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff, dropout)
                                     for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        tokens = self.embed(x.unsqueeze(-1)) + self.pos
        for block in self.blocks:
            tokens = block(tokens)
        return self.head(self.norm(tokens).mean(dim=1))

t0 = time.time()
transformer = TabularTransformer(X_nn.shape[1])
transformer = train_torch(transformer, X_nn, y_nn, X_va, y_va, epochs=SEQ_EPOCHS,
                          batch_size=1024, lr=1.5e-3)
proba_trf = predict_proba(transformer, X_va)
report("Transformer", y_va, proba_trf.argmax(1))
print(f"training time: {time.time() - t0:.1f}s")


### Visual check: attention between features

Each row is a feature asking a question; the columns show which features it listens to.

In [ ]:
# ---------------------------------------------------------------------
# Visual check: self-attention between the features
# ---------------------------------------------------------------------
plot_history(transformer, "Transformer training curve")

transformer.eval()
with torch.no_grad():
    xb = torch.from_numpy(np.ascontiguousarray(X_va[:512])).to(DEVICE)
    tokens = transformer.embed(xb.unsqueeze(-1)) + transformer.pos
    blk = transformer.blocks[0]
    xn = blk.norm1(tokens)
    att = blk.attn
    B, T, _ = xn.shape
    qkv = att.qkv(xn).reshape(B, T, 3, att.n_heads, att.d_head).permute(2, 0, 3, 1, 4)
    q, k, v = qkv[0], qkv[1], qkv[2]
    A = torch.softmax((q @ k.transpose(-2, -1)) / math.sqrt(att.d_head), dim=-1)
    A = A.mean(dim=1).mean(dim=0).cpu().numpy()      # average over heads and batch

plt.figure(figsize=(6.5, 5.5))
plt.imshow(A, cmap="viridis")
plt.xticks(range(len(FEATURES)), FEATURES, rotation=90, fontsize=7)
plt.yticks(range(len(FEATURES)), FEATURES, fontsize=7)
plt.xlabel("key (feature attended to)")
plt.ylabel("query (feature)")
plt.title("Transformer: mean self-attention between features")
plt.colorbar()
plt.tight_layout()
plt.show()


## Section 12 — Generative Adversarial Network (GAN)

So far every model has been *discriminative*: it draws a boundary between classes. A **GAN** (Goodfellow et al., 2014) is *generative*: it learns to **create** data.

### 12.1 The two-player game

Two networks are trained against each other:

* the **Generator** $G$ takes random noise $\mathbf{z}$ and produces a fake sample;
* the **Discriminator** $D$ sees a sample and must decide real or fake.

Formally it is a minimax game:

$$\min_G \max_D \; \mathbb{E}_{\mathbf{x}\sim p_{\text{data}}}\big[\log D(\mathbf{x})\big] + \mathbb{E}_{\mathbf{z}}\big[\log\big(1 - D(G(\mathbf{z}))\big)\big]$$

Gradient descent on $D$ (maximise: "spot the fakes") alternates with gradient descent on $G$ (minimise: "fool the discriminator"). At the equilibrium $G$ reproduces the true data distribution.

### 12.2 Conditional GAN

Our generator should create *specific* classes, so both networks also receive the **class label** (turned into a small embedding vector that is concatenated to the input). This is a **conditional GAN** (cGAN) — the same trick that powers class-conditional image generation.

### 12.3 Why does this help our metric?

Our problem is imbalanced: STAR is only 14 % of the data. The GAN lets us **manufacture extra STAR examples** that look statistically like the real ones, add them to the training set, and re-train the MLP. In a competition this is called **synthetic data augmentation** and is a well-known way to lift balanced accuracy.

### 12.4 Practical warnings

* **Mode collapse** — if $G$ finds one sample that always fools $D$, it may output that same sample forever. Watch the diversity of the generated samples.
* **D imbalance** — if $D$ becomes too strong, $G$'s gradients vanish. Using `betas=(0.5, 0.999)` and label smoothing (real label `0.9` instead of `1.0`) is the standard remedy.
* A GAN is *not* a classifier — it never sees the metric. It only makes the training soup richer.


In [ ]:
# =====================================================================
# Section 12 - Conditional GAN for minority-class augmentation
# =====================================================================
class Generator(nn.Module):
    """(noise z, class label) -> a plausible feature vector."""
    def __init__(self, n_features, n_classes, z_dim=32, hidden=256):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, 16)
        self.net = nn.Sequential(
            nn.Linear(z_dim + 16, hidden), nn.LeakyReLU(0.2),
            nn.Linear(hidden, hidden), nn.LeakyReLU(0.2),
            nn.Linear(hidden, n_features))

    def forward(self, z, labels):
        return self.net(torch.cat([z, self.label_emb(labels)], dim=-1))

class Discriminator(nn.Module):
    """(feature vector, class label) -> one real/fake logit."""
    def __init__(self, n_features, n_classes, hidden=256):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, 16)
        self.net = nn.Sequential(
            nn.Linear(n_features + 16, hidden), nn.LeakyReLU(0.2), nn.Dropout(0.2),
            nn.Linear(hidden, hidden), nn.LeakyReLU(0.2), nn.Dropout(0.2),
            nn.Linear(hidden, 1))

    def forward(self, x, labels):
        return self.net(torch.cat([x, self.label_emb(labels)], dim=-1))

Z_DIM = 32
BATCH = 512
generator = Generator(X_nn.shape[1], 3, z_dim=Z_DIM).to(DEVICE)
discriminator = Discriminator(X_nn.shape[1], 3).to(DEVICE)
opt_g = torch.optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()

X_pool = torch.from_numpy(np.ascontiguousarray(X_nn))
y_pool = torch.from_numpy(np.asarray(y_nn)).long()

t0 = time.time()
step_hist, d_hist, g_hist = [], [], []
for step in range(GAN_STEPS):
    # ---- 1) Update the Discriminator: real -> 1 (smoothed to 0.9), fake -> 0
    idx = torch.randint(0, len(X_pool), (BATCH,))
    real_x, real_y = X_pool[idx].to(DEVICE), y_pool[idx].to(DEVICE)
    z = torch.randn(BATCH, Z_DIM, device=DEVICE)
    fake_y = y_pool[torch.randint(0, len(y_pool), (BATCH,))].to(DEVICE)
    with torch.no_grad():
        fake_x = generator(z, fake_y)
    opt_d.zero_grad()
    d_loss = (bce(discriminator(real_x, real_y), torch.full((BATCH, 1), 0.9, device=DEVICE))
              + bce(discriminator(fake_x, fake_y), torch.zeros(BATCH, 1, device=DEVICE)))
    d_loss.backward()
    opt_d.step()

    # ---- 2) Update the Generator: it wants the Discriminator to say "1" (real)
    z = torch.randn(BATCH, Z_DIM, device=DEVICE)
    fake_y = y_pool[torch.randint(0, len(y_pool), (BATCH,))].to(DEVICE)
    opt_g.zero_grad()
    g_loss = bce(discriminator(generator(z, fake_y), fake_y), torch.ones(BATCH, 1, device=DEVICE))
    g_loss.backward()
    opt_g.step()

    if (step + 1) % 200 == 0:
        step_hist.append(step + 1)
        d_hist.append(float(d_loss))
        g_hist.append(float(g_loss))
        print(f"step {step + 1:5d}  D_loss={float(d_loss):.3f}  G_loss={float(g_loss):.3f}")
print(f"GAN training time: {time.time() - t0:.1f}s")

# --- Compare real vs generated STAR samples on the redshift axis ---------
STAR_IDX = 0
redshift_col = FEATURES.index("redshift")
real_star = X_nn[y_nn == STAR_IDX][:, redshift_col]
with torch.no_grad():
    synth_star = generator(torch.randn(40000, Z_DIM, device=DEVICE),
                           torch.full((40000,), STAR_IDX, dtype=torch.long, device=DEVICE))
synth_star = synth_star.cpu().numpy()

plt.figure(figsize=(9, 4))
plt.hist(real_star, bins=60, alpha=0.5, density=True, label="real STAR")
plt.hist(synth_star[:, redshift_col], bins=60, alpha=0.5, density=True, label="generated STAR")
plt.title("Conditional GAN: real vs generated STAR redshift")
plt.xlabel("Redshift (normalised)")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()

# --- Augment the minority class and re-train the MLP --------------------
X_aug = np.concatenate([X_nn, synth_star.astype("float32")], axis=0)
y_aug = np.concatenate([y_nn, np.full(len(synth_star), STAR_IDX, dtype=y_nn.dtype)], axis=0)
print("augmented training set:", X_aug.shape,
      "| STAR share: {:.1%}".format((y_aug == STAR_IDX).mean()))

t0 = time.time()
mlp_gan = MLP(X_aug.shape[1])
mlp_gan = train_torch(mlp_gan, X_aug, y_aug, X_va, y_va, epochs=MLP_EPOCHS,
                      batch_size=4096, lr=3e-3)
proba_mlp_gan = predict_proba(mlp_gan, X_va)
report("MLP + GAN aug", y_va, proba_mlp_gan.argmax(1))
print(f"training time: {time.time() - t0:.1f}s")


### Visual check: the adversarial game

Left: the two losses should keep hovering, not collapse. Right: generated STAR samples should sit on top of the real ones.

In [ ]:
# ---------------------------------------------------------------------
# Visual check: the adversarial game and what the GAN produces
# ---------------------------------------------------------------------
plot_history(mlp_gan, "MLP trained on GAN-augmented data")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(step_hist, d_hist, marker="o", label="discriminator loss")
axes[0].plot(step_hist, g_hist, marker="s", label="generator loss")
axes[0].set_xlabel("GAN step")
axes[0].set_ylabel("BCE loss")
axes[0].set_title("GAN: the two adversaries over time")
axes[0].legend()

from sklearn.decomposition import PCA
n_plot = 4000
real_star = X_nn[y_nn == STAR_IDX][:n_plot]
pca = PCA(n_components=2, random_state=SEED).fit(np.vstack([real_star, synth_star[:n_plot]]))
real_2d = pca.transform(real_star)
synth_2d = pca.transform(synth_star[:n_plot])
axes[1].scatter(real_2d[:, 0], real_2d[:, 1], s=6, alpha=0.5, label="real STAR")
axes[1].scatter(synth_2d[:, 0], synth_2d[:, 1], s=6, alpha=0.5, label="generated STAR")
axes[1].set_xlabel("PC 1")
axes[1].set_ylabel("PC 2")
axes[1].set_title("Real vs generated STAR (PCA)")
axes[1].legend()
plt.tight_layout()
plt.show()


## Section 13 — Ensembling and the Final Submission

We now have six trained models with genuinely different inductive biases:

| Model | What it is good at |
| :--- | :--- |
| Gradient-boosted trees | axis-aligned thresholds, handles the strong redshift cut |
| MLP | smooth non-linear interactions |
| RNN / LSTM | sequential reading of the features |
| Transformer | global pairwise feature relations |
| MLP + GAN augmentation | rebalanced class boundaries |

When models make **different kinds of mistakes**, averaging their probabilities is strictly better than picking the single best one. This is the **ensemble** idea, and it is how almost every Kaggle podium finishes.

### Blend, don't just vote

Instead of a hard majority vote we average the **softmax probabilities** with weights $w_m$:

$$\hat{\mathbf{p}} = \sum_m w_m\, \hat{\mathbf{p}}^{(m)}, \qquad \hat{y} = \arg\max_c \hat{p}_c$$

We search a small grid of weights and keep the combination with the best **validation balanced accuracy** — no test-set peeking.

### Your turn

* Try `NN_TRAIN_SIZE = len(X_tr)` and more epochs on a GPU: every model should improve.
* Replace the grid search with a logistic-regression *stacker* trained on the validation probabilities.
* After blending, **re-fit the winning models on train + validation** (80 % → 100 % of the data) before predicting the test set. This is standard practice and usually worth a final few tenths of a percent.


In [ ]:
# =====================================================================
# Section 13 - Ensemble and final submission
# =====================================================================
val_probas = {
    "GBDT": proba_gbdt,
    "MLP": proba_mlp,
    "RNN": proba_rnn,
    "LSTM": proba_lstm,
    "Transformer": proba_trf,
    "MLP+GAN": proba_mlp_gan,
}
print("--- Individual models on the validation set ---")
for name, proba in val_probas.items():
    report(name, y_va, proba.argmax(1))

# --- Small grid search over blend weights ------------------------------
ordered = ["GBDT", "MLP", "RNN", "LSTM", "Transformer", "MLP+GAN"]
val_stack = np.stack([val_probas[k] for k in ordered])          # (M, N, 3)

best_w, best_score = None, -1.0
for w in np.ndindex(*([3] * len(ordered))):                      # weights in {0, 1, 2}
    if sum(w) == 0:
        continue
    blend = np.tensordot(np.array(w, dtype="float32"), val_stack, axes=1)
    score = balanced_accuracy(y_va, blend.argmax(1))
    if score > best_score:
        best_score, best_w = score, w

print("\n--- Ensemble ---")
print("best weights:", {k: int(v) for k, v in zip(ordered, best_w)})
print(f"ensemble balanced_acc = {best_score:.4f}")
report("ENSEMBLE", y_va, np.tensordot(np.array(best_w, dtype="float32"), val_stack, axes=1).argmax(1))

# --- Predict the test set with the same blend --------------------------
test_stack = np.stack([
    gbdt.predict_proba(X_te),          # GBDT
    predict_proba(mlp, X_te),          # MLP
    predict_proba(rnn, X_te),          # RNN
    predict_proba(lstm, X_te),         # LSTM
    predict_proba(transformer, X_te),  # Transformer
    predict_proba(mlp_gan, X_te),      # MLP + GAN
])
test_blend = np.tensordot(np.array(best_w, dtype="float32"), test_stack, axes=1)
pred_te = test_blend.argmax(1)

submission = pd.DataFrame({"id": test_df["id"],
                           "class": [INV_CLASS_MAP[i] for i in pred_te]})
submission.to_csv("submission.csv", index=False)
print("\nsubmission.csv written:", submission.shape)
print(submission["class"].value_counts())
display(submission.head())


### Visual check: does the ensemble help?

The bars are the individual models, the dashed line is the blended ensemble, and the last panel shows the label mix we are about to submit.

In [ ]:
# ---------------------------------------------------------------------
# Visual check: does the ensemble beat the individual models?
# ---------------------------------------------------------------------
scores = [balanced_accuracy(y_va, val_probas[k].argmax(1)) for k in ordered]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
bars = axes[0].bar(ordered, scores, color="#4C72B0")
axes[0].axhline(best_score, color="red", ls="--", label=f"ensemble = {best_score:.4f}")
axes[0].set_ylabel("validation balanced accuracy")
axes[0].set_title("Individual models vs the ensemble")
axes[0].tick_params(axis="x", rotation=30)
axes[0].legend(fontsize=8)
for b, s in zip(bars, scores):
    axes[0].text(b.get_x() + b.get_width() / 2, s, f"{s:.3f}", ha="center", va="bottom", fontsize=8)

axes[1].bar(ordered, best_w, color="#DD8452")
axes[1].set_title("Blend weights found by the grid search")
axes[1].tick_params(axis="x", rotation=30)

sub_counts = pd.Series([INV_CLASS_MAP[i] for i in pred_te]).value_counts()
axes[2].bar(sub_counts.index, sub_counts.values, color="#55A868")
axes[2].set_title("Predicted test-class distribution")
plt.tight_layout()
plt.show()


## Section 14 — Wrapping Up

### What you should observe

| Model | Typical validation balanced accuracy | Notes |
| :--- | ---: | :--- |
| KNN (k=100, Section 3) | ≈ 0.93 | the starting point on the leaderboard |
| Gradient-boosted trees | ≈ 0.950 | strongest single "classical" model |
| MLP | ≈ 0.940 | needs a GPU + a few epochs |
| RNN / LSTM | ≈ 0.930–0.945 | sequential view of a table is a bit artificial |
| Transformer | ≈ 0.930–0.945 | shines on sequences, not on 14 independent columns |
| MLP + GAN augmentation | ≈ 0.940–0.950 | rebalancing helps the rare STAR class |
| **Weighted ensemble** | **≈ 0.955+** | usually the best of the lot |

*(Exact numbers vary with the seed and how long you train; the ordering of the rows is very stable.)*

### The two lessons that matter

1. **Match the architecture to the data.** For tabular data, gradient-boosted trees beat every neural network we threw at them, at a fraction of the cost. Deep learning is not automatically better — it is better when the data has structure that matches its inductive bias (sequences, images, language).
2. **Optimise the metric you are graded on.** Switching from accuracy to *balanced* accuracy, weighting the loss and rebalancing with a GAN all pushed the rare STAR class — which is exactly what the leaderboard measures.

### Extensions worth trying

* **Out-of-fold target encoding** for `spectral_type` / `galaxy_population` (a known leaderboard trick on this dataset).
* **Stacking**: feed the validation probabilities of all base models into a small logistic regression.
* **Seed averaging**: train each neural net 3 times with different seeds and average.
* **k-fold** instead of a single split, for a more reliable estimate of the score.
* Swap `HistGradientBoostingClassifier` for `LightGBM` / `XGBoost` / `CatBoost` — on tabular data they are the state of the art.

### Where each idea still lives today

* MLP → recommendation systems, tabular deep learning, the head of every transformer.
* RNN / LSTM → time series, sensor streams, speech; the ancestor of modern sequence models.
* Attention / Transformer → GPT, BERT, ViT, protein folding — the dominant architecture of this decade.
* GANs → image synthesis, super-resolution, medical data augmentation; also the intellectual ancestor of diffusion models.

You have now gone all the way from a one-line decision rule to a trained, ensembled deep-learning pipeline — on the very same photons from the SDSS telescope.
